# 22 — Portfolio Credit: CDOs & Nth-to-Default

- **Binomial loss model** for credit baskets
- **Gaussian LHP** (Large Homogeneous Portfolio) approximation
- **CDO tranches**: equity, mezzanine, senior
- **Nth-to-default** baskets
- Correlation sensitivity and base correlation
- Copula models

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ql_jax.experimental.credit.loss_models import (
    CreditBasket, binomial_loss_model, gaussian_lhp_loss, NthToDefault)
from ql_jax.experimental.credit.cdo import (
    CdoTranche, SyntheticCdo, expected_tranche_loss)
from ql_jax.math.copulas import gaussian_copula, clayton_copula

---
## 1. Credit Basket

In [ ]:
N_NAMES = 50
DEFAULT_PROB = 0.05    # 5% 5Y default prob, each name
RECOVERY = 0.40
CORRELATION = 0.30

basket = CreditBasket(
    default_probs=jnp.full(N_NAMES, DEFAULT_PROB),
    recovery_rates=jnp.full(N_NAMES, RECOVERY),
    notionals=jnp.full(N_NAMES, 1.0 / N_NAMES)  # normalized
)

print(f"Portfolio: {N_NAMES} names, PD={DEFAULT_PROB:.0%}, R={RECOVERY:.0%}, ρ={CORRELATION:.0%}")

---
## 2. Loss Distribution

In [ ]:
loss_levels, loss_probs = binomial_loss_model(basket, CORRELATION)

plt.figure(figsize=(10, 5))
plt.bar(np.array(loss_levels)[:20] * 100, np.array(loss_probs)[:20] * 100, width=0.8)
plt.xlabel('Portfolio Loss (%)')
plt.ylabel('Probability (%)')
plt.title(f'Loss Distribution (ρ={CORRELATION:.0%})')
plt.grid(True, alpha=0.3)
plt.show()

exp_loss = float(jnp.sum(loss_levels * loss_probs))
print(f"Expected portfolio loss: {exp_loss*100:.2f}%")

---
## 3. CDO Tranches

In [ ]:
tranches = [
    CdoTranche(attachment=0.00, detachment=0.03),   # equity
    CdoTranche(attachment=0.03, detachment=0.07),   # mezzanine
    CdoTranche(attachment=0.07, detachment=0.15),   # senior
    CdoTranche(attachment=0.15, detachment=1.00),   # super-senior
]

cdo = SyntheticCdo(portfolio_notional=1.0, tranches=tranches, recovery_rate=RECOVERY)

print(f"{'Tranche':<15s} {'Attach':>8s} {'Detach':>8s} {'Exp Loss':>10s}")
print("-" * 45)
for i, t in enumerate(tranches):
    el = float(expected_tranche_loss(
        jnp.full(N_NAMES, DEFAULT_PROB), RECOVERY,
        t.attachment, t.detachment, CORRELATION))
    width = t.detachment - t.attachment
    name = ['Equity', 'Mezzanine', 'Senior', 'Super-Senior'][i]
    print(f"{name:<15s} {t.attachment:>7.0%} {t.detachment:>7.0%} {el/width*100:>9.2f}%")

---
## 4. Gaussian LHP Approximation

In [ ]:
lhp_equity = float(gaussian_lhp_loss(DEFAULT_PROB, RECOVERY, CORRELATION, 0.0, 0.03))
lhp_mezz   = float(gaussian_lhp_loss(DEFAULT_PROB, RECOVERY, CORRELATION, 0.03, 0.07))
lhp_senior = float(gaussian_lhp_loss(DEFAULT_PROB, RECOVERY, CORRELATION, 0.07, 0.15))

print(f"LHP Equity expected loss  : {lhp_equity/0.03*100:.2f}%")
print(f"LHP Mezz expected loss    : {lhp_mezz/0.04*100:.2f}%")
print(f"LHP Senior expected loss  : {lhp_senior/0.08*100:.2f}%")

---
## 5. Correlation Sensitivity

In [ ]:
corrs = np.linspace(0.01, 0.80, 40)
eq_losses = [float(gaussian_lhp_loss(DEFAULT_PROB, RECOVERY, c, 0.0, 0.03)) / 0.03 * 100
             for c in corrs]
sr_losses = [float(gaussian_lhp_loss(DEFAULT_PROB, RECOVERY, c, 0.07, 0.15)) / 0.08 * 100
             for c in corrs]

plt.figure(figsize=(10, 6))
plt.plot(corrs * 100, eq_losses, 'r-', linewidth=2, label='Equity [0-3%]')
plt.plot(corrs * 100, sr_losses, 'b-', linewidth=2, label='Senior [7-15%]')
plt.xlabel('Correlation (%)')
plt.ylabel('Expected Tranche Loss (%)')
plt.title('Tranche Loss vs Correlation (LHP)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 6. Nth-to-Default Basket

In [ ]:
basket5 = CreditBasket(
    default_probs=jnp.array([0.05, 0.04, 0.06, 0.03, 0.05]),
    recovery_rates=jnp.full(5, 0.40),
    notionals=jnp.full(5, 0.20)
)

for nth in [1, 2, 3]:
    ntd = NthToDefault(basket=basket5, nth=nth, premium_rate=0.01, maturity=5.0)
    dl = float(ntd.default_leg_value(CORRELATION, discount_factor=np.exp(-0.03*5)))
    print(f"{nth}-th to default, default leg : {dl:.6f}")

---
## 7. AD: Correlation Delta

In [ ]:
def equity_loss_fn(corr):
    return gaussian_lhp_loss(DEFAULT_PROB, RECOVERY, corr, 0.0, 0.03)

corr_delta = float(jax.grad(equity_loss_fn)(CORRELATION))
print(f"dEquityLoss/dCorrelation : {corr_delta:.6f}")
print(f"Per 1% corr move        : {corr_delta * 0.01:.6f}")

---
## 8. Exercises

1. **Base correlation**: Given market tranche spreads, imply the base correlation for each detachment.
2. **Copula comparison**: Replace the Gaussian copula with Clayton and compare loss distributions.
3. **Heterogeneous portfolio**: Use different PDs per name and compare to the homogeneous LHP.

---
*End of Notebook 22*